# Example - Identification of System Model Parameters from Data

In [ ]:
import os

import casadi as cas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cas_models.discrete_time.models import (
    StateSpaceModelDT,
    StateSpaceModelDTARXSISO,
)
from cas_models.discrete_time.simulate import (
    make_n_step_simulation_function_from_model,
    prediction_errors_ms,
)
from cas_models.transformations import connect_systems
from plot_utils import make_io_plot

In [ ]:
data_dir = "data"
os.listdir(data_dir)

In [ ]:
filename = 'data_center_sim_results.csv'

id_data = pd.read_csv(os.path.join(data_dir, filename))
id_data.info()

In [ ]:
id_data.head()

In [ ]:
input_names = ["CRAC1_flow", "CRAC2_flow"]
output_names = ["R11_intake"]

fig, axes =make_io_plot(id_data, input_names, output_names)
plt.tight_layout()
plt.show()

In [ ]:
# Construct a 2-input, 1-output system model with symbolic parameters
sys1 = StateSpaceModelDTARXSISO(na=2, nb=1, nk=1, input_name="CRAC1_flow", output_name="y1")
sys2 = StateSpaceModelDTARXSISO(na=2, nb=1, nk=1, input_name="CRAC2_flow", output_name="y2")

# Sum the outputs into one output
connections = {"R11_intake": ["y1", "y2"]}
sys_model = connect_systems(
    [sys1, sys2],
    connections,
    StateSpaceModelDT,
    output_names=["R11_intake"],
    name="R11_intake"
)

sys_model.describe()

In [ ]:
u_nop = id_data[input_names].iloc[0].to_numpy()
y_nop = id_data[output_names].iloc[0].to_numpy()
u_nop, y_nop

In [ ]:
# Prepare input/output data
t = id_data.index.to_numpy()
nT = t.shape[0] - 1
simulate = make_n_step_simulation_function_from_model(sys_model, nT)
simulate

In [ ]:
# Initial guess of parameters
params_init = {
    'sys1_a_0': -1.23,
    'sys2_a_0': -1.78,
    'sys1_a_1': 0.307,
    'sys2_a_1': 0.785,
    'sys1_b': -0.278,
    'sys2_b': -0.012
}

# From OLS, see analysis/system-identification-four-racks-dr.ipynb
# [-1.23444165,  0.30742272, -0.27760564]
# [-1.78060413,  0.78521152, -0.01211544]

U = id_data[input_names].to_numpy() - u_nop
Y_m = id_data[output_names].to_numpy() - y_nop

x0 = cas.DM.zeros(sys_model.n, 1)
X, Y = simulate(cas.DM(t), U[:nT], x0, *params_init.values())
X = np.asarray(X)
Y = np.asarray(Y)
assert X.shape == (nT+1, sys_model.n)
assert Y.shape == (nT+1, sys_model.ny)

y_pred = pd.DataFrame(Y + y_nop, index=id_data.index, columns=sys_model.output_names)

fig, axes = make_io_plot(id_data, input_names, output_names, y_pred=y_pred)
plt.tight_layout()
plt.show()

y_pred_err = y_pred - Y_m
rmse = float(np.mean(np.sqrt(y_pred_err ** 2)))
print(f"{rmse = :.3f}")

In [ ]:
def estimate_system_parameters(
    model, t, U, Y_m, x0=None, X_init=None, n_steps=10, params_init=None, param_bounds=None
):

    # Use nonlinear optimisation to estimate all the parameters of the model
    opti = cas.Opti()

    # Decision variables: one for each model parameter
    params = {name: opti.variable() for name in model.params}

    # Model output simulated with these parameters, from a zero initial state
    if x0 is None:
        x0 = cas.DM.zeros(model.n, 1)
        # TODO: Allow x0 to be a decision variable

    # Use multiple-shooting method
    errors, _ = prediction_errors_ms(opti, model, t, U, Y_m, x0, params, n_steps, X_init=X_init)

    # Objective: sum of squared errors between the data and the simulated output
    opti.minimize(cas.sumsqr(errors))

    # Constraints on parameter values
    if param_bounds is not None:
        for name, (lb, ub) in param_bounds.items():
            p = params[name]
            if lb is not None:
                opti.subject_to(p >= lb)
            if ub is not None:
                opti.subject_to(p <= ub)

    # Initial guesses
    if params_init is not None:
        for name, value in params_init.items():
            opti.set_initial(params[name], value)

    # Limited-memory Hessian: same solution, about half the time of the exact one
    opti.solver("ipopt", {}, {"hessian_approximation": "limited-memory"})
    sol = opti.solve()

    params_est = {name: float(sol.value(var)) for name, var in params.items()}
    return params_est

params_est = estimate_system_parameters(
    sys_model,
    t,
    U[:-1, :],
    Y_m,
    x0=None,
    X_init=None,
    n_steps=20,
    #params_init=params_init,
    param_bounds=None
)

In [ ]:
pd.concat([
    pd.Series(params_init, name="params_init"), 
    pd.Series(params_est, name='params_est')
], axis=1)

In [ ]:
x0 = cas.DM.zeros(sys_model.n, 1)
X, Y = simulate(cas.DM(t), U[:nT], x0, *params_est.values())
X = np.asarray(X)
Y = np.asarray(Y)
assert X.shape == (nT+1, sys_model.n)
assert Y.shape == (nT+1, sys_model.ny)

y_pred = pd.DataFrame(Y + y_nop, index=id_data.index, columns=sys_model.output_names)

fig, axes = make_io_plot(id_data, input_names, output_names, y_pred=y_pred)
plt.tight_layout()
plt.show()

y_pred_err = y_pred - Y_m
rmse = float(np.mean(np.sqrt(y_pred_err ** 2)))
print(f"{rmse = :.3f}")